# Ensembles and Random Forests
## DS-3001: Foundations of Machine Learning

Content adapted from Terence Johnson (UVA)

### Loading Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # For changing directory

# To mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_DS_3001_folder = '/content/drive/MyDrive/DS-3001/03_Tuning_Testing_Evaluating'
# path_to_DS_3001_folder = ''

# Update the path to your folder for the class
# Where you stored the data from the previous notebook
os.chdir(path_to_DS_3001_folder)

## How can we use bootstrapping to improve our models?

- The point of the bootstrap lecture was to introduce the idea and power of **resampling**

- **Your data aren't just one set of data**:
  - They are a deep reservoir of information that can be used to quantify all kinds of uncertainty

- Can these principles be applied directly to solving machine learning problems?

## Bootstrapping to Advance Decision Trees

### Reviewing Decision Trees

#### What is a decision tree?

- A *decision tree* is
  1. A set of *decision nodes* that represent choices.
  2. A set of edges that represent *decisions* at each decision node (data-driven choices).
  3. A set of *terminal nodes* or *outcomes* (predictions or courses of action).

- The goal is to build a decision tree using data that predicts outcomes (classification or regression) for future cases.

- Here's an example decision tree for classifying animals ([image source](https://towardsai.net/p/programming/decision-trees-explained-with-a-practical-example-fe47872d3b53)):

![](https://cdn-images-1.medium.com/max/824/0*J2l5dvJ2jqRwGDfG.png)

#### How do decision trees work?

- Building a decision iterates on these steps in **classification** tasks:

  1. For all of the variables, determine the split for each that minimizes **Gini impurity** = $\sum_{k=1}^{K} p_k * (1 - p_k)$ .

  2. Pick the variable that achieves the lowest Gini impurity overall. This creates a single split in the tree. (We might use this variable again to build the tree.)

  3. Repeat steps 1 and 2 *for each of the sub-populations created by our previous splits* until the impurity is 0 for the remaining sub-populations (they are perfectly separated by outcome) or we reach some stopping rule.

- How about for **regression** tasks where the outcome is continuous? It's very similar, but instead of gini impurity we use Mean Squared Error (MSE) or Sum of Squared Error (SSE):

  1. Pick a split point, $s$ for the numeric predictor/regressor $x$

  2. On either side of the split, compute the mean values of the outcome, $\bar{y}_L(s)$ and $\bar{y}_R(s)$

  3. Compute the SSE for the split $s$:
    \begin{gather}
    \text{SSE}(s) = \underbrace{\sum_{x<s} (y(x) - \bar{y}_L(s))^2}_{\text{Left SSE}} + \underbrace{\sum_{x \ge s} (y(x)-\bar{y}_R(s))^2}_{\text{Right SSE}}
    \end{gather}

  4. Choose $s$ to minimize `SSE(s)`

- As we move down the tree, the goal is to reduce the `SSE` along each split/path. Stopping rules are critical with numeric outcomes.  

- The classification tree is implemented in Sklearn as [sklearn.tree.DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)

- The regression tree is implemented in Skelarn as [sklearn.tree.DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)

#### What are the pitfalls of decision trees?

- They're very prone to **overfitting**. Without stopping rules, they will learn the training data set exaclty.

## How can we compare models without dimensions? $R^2$

### Why we need a new method of model comparison:

- We need to add another tool of model evaluation technology

- In many contexts, we're comparing models across types/specifications or even across datasets

- For quick comparison, we'd like a metric that is **dimensionless**: Its units cancel out, so it just provides a context-free summary of fit

- For regression, the most popular version of this is called the **$R$-squared** measure (the **coefficient of determination**)

### Deriving the $R^2$ measure:

1. Imagine you made your predictions using only the mean as a predictor. Your SSE would be: $$SSE_{mean} = \sum_{i=1}^N (y_i - \bar{y})^2 $$

2. Imagine you made your predictions using your full model, $\hat{y}_i = m(x_i,b)$. Your SSE would be: $$SSE_{model} = \sum_{i=1}^N (y_i - \hat{y}_i)^2 $$

3. What is the proportional decrease in $SSE$, moving from the simple model to your more complex model?

$$
R^2 = \dfrac{SSE_{mean} - SSE_{model}}{SSE_{mean}} = 1- \dfrac{SSE_{model}}{SSE_{mean}}
$$

### What values can $R^2$ take on and what do they mean?

We have some rules of thumb for interpreting it:

- If $R^2 = 0$, the model does no better than prediction using the (training) mean
- If $R^2 = 1$, the model fits perfectly
- If $R^2 < 0$, the model does worse than prediction using the (training) mean. This can happen when you compute $R^2$ using a train/test split or cross validation, and evaluate it on the test set.


### Data Set for Today

- Let's move towards fitting some regression decision trees and see how we can use bootstrapping.

- We'll look at a housing data set where we want to predict `sale_price` from a variety of features of a single property.

- We first want to see how a decision tree performs on this data set.

In [ ]:
# Loading in our data set
house_df = pd.read_csv('./data/pierce_county_house_sales.csv')

# Create an age variable
house_df['age'] = max(house_df['year_built']) - house_df['year_built']

# Select variables of interest to look at
vars = [
    "attached_garage_square_feet", "attic_finished_square_feet",
    "basement_square_feet", "bathrooms",
    "bedrooms", "detached_garage_square_feet",
    "fireplaces", "house_square_feet",
    "stories", "age", "sale_price"
]

# Subset to get the variables of interest
house_df = house_df.loc[:,vars]

# Look at the subset data frame
house_df.head()

In [ ]:
house_df.describe()

In [ ]:
from sklearn.model_selection import train_test_split

# Import for a tree
from sklearn import tree

# 1. Split the data into train/test
# 2. Create a function to fit a regression tree on a bootstrapped sample


In [ ]:
# Use your new function to fit a bootstrapped tree
# and gather the predictions


In [ ]:
# Fit a second bootstrapped tree and gather predictions


**How do the predictions compare?**

In [ ]:
sns.scatterplot(x = y_hat_1, y = y_hat_2)
plt.xlabel('Predictions from First Bootstrapped Model')
plt.ylabel('Predictions from Second Bootstrapped Model')
plt.title('Comparing Predictions from Bootstrapped Models')
plt.show()

In [ ]:
# Creating a data frame with our predictions as columns
tdf = pd.DataFrame({'y_hat_1':y_hat_1, 'y_hat_2':y_hat_2})

# Calculating a correlation matrix
tdf.corr()

- These estimates are pretty similar, but also pretty different, even though the test set is the same

## An Example of Model Fragility

- Seeing that our predictions vary somewhat significantly depending on the training data can be pretty bad:
  - For trees specifically, the predictor variables change and cutpoints change depending on our training data set.
  - This doesn't look like it is capturing real, robust features of the data

- **This issue could arise for any model with hyperparameters:**
  - We are highlighting decision trees here because it is a particular offender

- It appears that the out-of-the-box decision tree is just a bit too sensitive to the underlying features of the data

- If this kind of bootstrap resampling reveals a problem with the decision tree model, **maybe it also suggests a solution?**

# Bootstrapping Our Models

## Ensembles and Bootstrap Aggregation

- **What we've done so far:**
  - We've seen that it's easy to take bootstrap samples of the original training data and fit decision trees on each sample.

- **Creating an ensemble of models:**

  - Using this set up, we can estimate a *set* of decision trees for a large number of bootstrap samples from the original data
  - This is called an **ensemble**
  - We then average the predictions of each bootsrapped model to get a predictor that is more robust

- **Terms for ensemble models:**
  - This can be called many things:
    - **bootstrap aggregation**
    - **bagging**
    - **model averaging**
  - What it allows us to do:
    - Take flawed models trained on random subsets of the data and combine them to improve overall performance

- The general name for this technique is **ensemble learning**

- Why? The weak law of large numbers (WLLN) implies convergence of averages, so let's average over models.

## Fitting an Ensemble

<div>
<img src="https://github.com/DS3001/ensembles/blob/main/ensemble_construction.jpeg?raw=true" width="700"/>
</div>

## Predicting with an Ensemble

<div>
<img src="https://github.com/DS3001/ensembles/blob/main/ensemble_prediction.jpeg?raw=true" width="700"/>
</div>

## Example of Ensemble Learning: 100 Trees for Prediction

- **Task**:
  - Predict the sale price of a home using an ensemble of decision trees.
  - We'll use 100 trees built on 100 random bootstrap samples of the original data
  - **Why only 100 trees?** This depends on the application, but we want it to run reasonably quickly for class purposes, so we'll constrain the max depth to a moderate value.

- **Goal:**
  - Compare the $R^2$ and variance of the individual trees versus the ensemble.
  - Some packages will automate this, but we'll do it "by hand" to be clear about how the idea is actually being implemented.

In [ ]:
# Fit 100 bootstrapped trees and compare performance to the ensemble model

# Import tqdm for a progress bar
from tqdm import tqdm


In [ ]:
# Plot the distribution of the R^2 for all individual trees
ax = sns.kdeplot(
    x = R_sq, fill = True,
    color = 'dodgerblue', alpha = 0.1,
    label = r'$R^2$ Individual'
)
ax.axvline(
    x = R_sq_ensemble,
    label = r'$R^2$ Ensemble',
    color = 'firebrick'
)

plt.title('Comparison Between Individual Trees vs Ensemble Predictor Performance')
plt.legend()
plt.show()

# What was the maximum R^2 for individual trees
print('The Maximum R^2 for Individual Trees:', np.quantile(R_sq, 1))
print('The R^2 for the Ensemble Model:', R_sq_ensemble)


## Advantage of the Ensemble Method for this Example

- Given this specific example, we got an $R^2$ of 0.36. This is better than any of the individual learners that had a maximum $R^2$ of 0.348.

- **The ensemble predictor is stable**:
  - It is averaging over many draws of data, and because of the WLLN, adding more draws won't change its value a significant amount.
  - We won't have a different predictor every day.

- This didn't require a "complicated" strategy, just a really clever one: Averaging models over our bootstrap resampling

## Understanding the Ensemble versus Individual Learners

- *The performance of the ensemble average is not the average of ensemble performance*
  - In other words, the $R^2$ of the ensemble is not the average $R^2$ of the individual trees.

- When we bootstrap, **the ensemble typically performs better than the individual** by decreasing the variance:
  - Higher accuracy overall and less variance than the individual learners.

- **A big payoff is a massive reduction in prediction variance**:
  - Without increasing bias, we're making much less noisy predictions overall.

- This is a big step towards "fixing" decision trees.

- **The intuition of ensemble learning:**
  - Each weak learner (tree) learns a bit of the true signal but also some noise
  - Averaging many models that each capture a bit of signal **keeps the signal and averages away the noise**, resulting in a high quality model.

## How does this apply to classification tasks?

- For regression, we averaged the predictions of each of the ensemble members to get an ensemble prediction.

- For classification, **we have each of the ensemble members "vote" on a category, and the category with the most votes from all the ensemble members wins.**